In [1]:
import pandas as pd
import numpy as np
import random
import pyomo.environ as pyo
from pyomo.environ import *
from pyomo.environ import SolverFactory
import yfinance as yf
from pathlib import Path
import os
import matplotlib.pyplot as plt


In [2]:
anos = ['2016-12-31', '2017-12-31', '2018-12-31', '2019-12-31', '2020-12-31', '2021-12-31', '2022-12-31', '2023-12-31', '2024-12-31', '2025-12-31', '2026-12-31']
print(anos)

['2016-12-31', '2017-12-31', '2018-12-31', '2019-12-31', '2020-12-31', '2021-12-31', '2022-12-31', '2023-12-31', '2024-12-31', '2025-12-31', '2026-12-31']


## LER OS RETORNOS, SCORE e SELIC

In [3]:

# Retornos

dfs = {}
for ano in anos:
    ano_usado = ano.split('-')[0]
    ano_usado_mais_um = int(ano_usado)+1
    # print(ano_usado, "-", type(ano_usado))

    nome = f'df_ativos_{ano_usado}.csv'
    # print(nome, "-", type(nome))
    # caminho = Path('retornos_ativos_otimizacao') / nome
    dfs[ano_usado] = pd.read_csv('../../../novos-dados-retornos/otimizacao/'+nome)
    dfs[ano_usado]['date'] = pd.to_datetime(dfs[ano_usado]['date'])
    dfs[ano_usado] = dfs[ano_usado].set_index('date')
    # print(dfs[ano_usado])
    dfs[ano_usado] = dfs[ano_usado].drop(columns=dfs[ano_usado].columns[(dfs[ano_usado] == 0).all()])
    # dfs[ano_usado] = dfs[ano_usado].loc[str(ano_usado_mais_um)]
    ativos_manter = (dfs[ano_usado]==0.0).mean()
    c_manter = ativos_manter[ativos_manter<0.5].index
    dfs[ano_usado] = dfs[ano_usado][c_manter]

    print(ano,'-',len(dfs[ano_usado].columns))
    

2016-12-31 - 68
2017-12-31 - 69
2018-12-31 - 75
2019-12-31 - 77
2020-12-31 - 80
2021-12-31 - 90
2022-12-31 - 98
2023-12-31 - 99
2024-12-31 - 99
2025-12-31 - 99
2026-12-31 - 100


## Hiperparâmetros

In [4]:
vb_cardinalidade_max = 20
# vb_cardinalidade_min = 10
vb_peso_maximo = 0.10
# vb_peso_minimo = 0.02
vb_theta = 0.90

### Fazendo otimização ano a ano e comparando com o proximo ano

In [6]:
y = []

carteiras_anuais = {}

melhor_pesos = None
historico = []
carteiras_criadas = []


for an in anos:
    ano = an.split("-")[0]
    print("COMEÇANDO ANO NOVO: ",ano)
    #deletar modelo
    if 'model' in locals():
        del model
        print('Modelo Antigo deletado \n Iniciando Novo')
    else:
        print("Nao consta model")
        pass
    
    # Retornos só para filtrar colunas
    df_usado = dfs[str(ano)]
    retorno_usado = df_usado.copy()    
        #SCORES MF e PIO------------
    score_mf_todos_anos = pd.read_excel(f'../../../score-raspagem/magic_formula/resultado/{int(ano)-1}.xlsx')
    score_mf = score_mf_todos_anos[['Unnamed: 0', 'magic_formula']].set_index('Unnamed: 0').T
    
    score_pio_todos_anos = pd.read_excel(f'../../../score-raspagem/piotroski/resultado/{int(ano)-1}.xlsx')
    score_pio = score_pio_todos_anos[['Unnamed: 0', 'p_norm']].set_index('Unnamed: 0').T
    
    ativos = retorno_usado.columns.intersection(score_pio.columns).intersection(score_mf.columns)
    print("ATIVOS: ",ativos)
    score_usado_mf = score_mf[ativos].copy()
    score_usado_pio = score_pio[ativos].copy()

    print("Score Atualizado: ",int(ano)-1)


    print("-----")
    

    # if df_usado == 'df_ativos_2015':
    #     continue
    # else:
    print(f"{int(ano)-1} TAMANHOS: Retorno {len(ativos)}, Score PIO {len(score_usado_pio.columns)}, Score MF {len(score_usado_mf.columns)}")
    print("# ------ CRIAÇÃO DO MODELO")
    print("## Carteira criada para o ano: ",int(ano))

    # # print("## UTILIZANDO SCORE DO ANO DE: ",ano)
    # print("## UTILIZANDO DADOS DE RETORNO DE: ",str(ano_um))
    # # print("## UTILIZANDO EXCESSO DO ANO DE: ",ano_um)
    # print("## UTILIZANDO SIGMAS DO ANO DE: ",ano_um)

    model = pyo.ConcreteModel()

    #---------VARIÁVEIS-----------
    model.nome_ativos = pyo.Set(initialize = ativos)
    model.ativos = pyo.RangeSet(0, len(ativos)-1)
    # model.dias = pyo.RangeSet(0, len(retorno_usado)-1)
    # model.retornos_ativos = pyo.Param(model.dias, model.ativos, initialize=lambda model,dia, ativo: retorno_usado.iloc[dia, ativo])    
    # model.theta = pyo.Param(initialize=vb_theta)
    model.score_p = pyo.Param(model.ativos, initialize=lambda model,a: score_usado_pio.iloc[0,a])
    model.score_m = pyo.Param(model.ativos, initialize=lambda model,a: score_usado_mf.iloc[0,a])
    model.cardinalidade_valor_max = pyo.Param(initialize=vb_cardinalidade_max)
    # model.cardinalidade_valor_min = pyo.Param(initialize=vb_cardinalidade_min)
    model.peso_maximo = pyo.Param(initialize=vb_peso_maximo)
    # model.peso_minimo = pyo.Param(initialize=vb_peso_minimo)
    model.x = pyo.Var(model.ativos, bounds=(0,1))
    model.y = pyo.Var(model.ativos, within=pyo.Binary)
    # model.excesso = pyo.Param( model.ativos , initialize = lambda model,a: excesso_usado.mean().iloc[a])
    # model.sigma = pyo.Param(model.ativos, model.ativos, initialize = lambda model,a,b: sigma_usado.iloc[a,b])
    # model.s = pyo.Param(initialize = 2, mutable=True)
    # model.r = pyo.Var(within=pyo.NonNegativeReals)
    
    for i in range(3):
        setattr(model,f'obj{i}_mais',pyo.Var(domain=NonNegativeReals)) #bounds=(0,0.1),
        setattr(model,f'obj{i}_menos',pyo.Var(domain=NonNegativeReals))
    #-------------------------------------- FUNÇÕES

    #=============================
    # Função Objetivo
    #=============================

    def func_objetivo_1(model):
       
        return model.obj1_menos + model.obj2_menos
    model.objetivo = pyo.Objective(rule=func_objetivo_1, sense=pyo.minimize)

    #=============================
    # RESTRIÇÕES
    #=============================

    # # OBJ 1 2 3 - Rest 1 2 3
    # def retorno_restr_rule(model,diad):
    #     return sum(model.x[a]*model.retornos_ativos[dia,a] for a in model.ativos for dia in model.dias) + model.obj0_menos - model.obj0_mais >= model.retornos_ibov[diad]
    # model.retorno_restr = pyo.Constraint(model.dias,rule=retorno_restr_rule)

    def piotroski_restr_rule(model):
        return sum(model.x[a]*model.score_p[a] for a in model.ativos) + model.obj1_menos - model.obj1_mais >= 0.9
    model.piotroski_rest = pyo.Constraint(rule=piotroski_restr_rule)

    def mf_restr_rule(model):
        return sum(model.x[a]*model.score_m[a] for a in model.ativos) + model.obj2_menos - model.obj2_mais >= 0.9
    model.mf_restr = pyo.Constraint(rule=mf_restr_rule)

    #REstricao 1 x só ativa se y = 1
    def restr_vinculo_x_y(model, a):
        return model.x[a] <= model.y[a]
    model.const_restr_vinculo_x_y = pyo.Constraint(model.ativos, rule=restr_vinculo_x_y)

    #peso maximo por acao
    def rule_peso_maximo(model, a):
        # return model.x[a] <= 1/model.cardinalidade_valor
        return model.x[a] <= model.peso_maximo
    model.const_peso_maximo = pyo.Constraint(model.ativos, rule=rule_peso_maximo)

    # #peso minimo por acao
    # def rule_peso_minimo(model, a):
    #     return model.x[a] >= model.peso_minimo * model.y[a]  # se y=1, então x >= 0.05
    # model.const_peso_minimo = pyo.Constraint(model.ativos, rule=rule_peso_minimo)

    #Restrição 2 soma peso 1
    def soma_peso_1(model):
        return sum(model.x[a] for a in model.ativos) == 1
    model.const_soma_peso_1 = pyo.Constraint(rule=soma_peso_1)


    # def cardinalidade_min(model):
    #     return sum(
    #         model.y[a] for a in model.ativos
    #         ) >= model.cardinalidade_valor_min
    # model.const_cardinalidade_total_min = pyo.Constraint(rule=cardinalidade_min)

    def cardinalidade_max(model):
        return sum(
            model.y[a] for a in model.ativos
            ) <= model.cardinalidade_valor_max
    model.const_cardinalidade_total_max = pyo.Constraint(rule=cardinalidade_max)


    # NOTEBOOOK
    # opt = SolverFactory('cplex', executable='C:\\CPLEX_Studio2211\\cplex\\bin\\x64_win64\\cplex.exe')

    # PC
    opt = SolverFactory('cplex', executable='C:\\Program Files\\IBM\\ILOG\\CPLEX_Studio_Community222\\cplex\\bin\\x64_win64\\cplex.exe')
    res = opt.solve(model,tee=False)
    print("Situação SOLVER: ",res.solver.termination_condition)
    if (res.solver.status == SolverStatus.ok) and (res.solver.termination_condition == TerminationCondition.optimal): 
        melhor_pesos = {list(model.nome_ativos.data())[a]: pyo.value(model.x[a]) for a in model.ativos}
        pio = {sum(pyo.value(model.x[t])*pyo.value(model.score_p[t]) for t in model.ativos)}
        mf = {sum(pyo.value(model.x[t])*pyo.value(model.score_m[t]) for t in model.ativos)}
        carteiras_anuais[int(ano)] = {
            'pesos':  melhor_pesos,
            # 'sharpe_anual': s_lo*np.sqrt(252),
        }
        print(f"{ano} -> {melhor_pesos}")
        print(f"{int(ano)-1} Piotroski -> {pio}")
        print(f"Valores obj Menos: {pyo.value(model.obj1_menos), pyo.value(model.obj1_mais)}")
        print(f"{int(ano)-1} Magic Formula -> {mf}")
        print(f"Valores obj Menos: {pyo.value(model.obj2_menos), pyo.value(model.obj2_mais)}")
        print("+-="*30)
        if 'model' in locals():
            del model
            print('DELETADO DPS DO WHILE')
        else:
            print("Nao consta model")
            pass
    else:
        print("PROblema no ANO: ", ano)
        break




COMEÇANDO ANO NOVO:  2016
Nao consta model
ATIVOS:  Index(['ABEV3', 'ALOS3', 'ALPA4', 'ANIM3', 'AXIA3', 'AZZA3', 'B3SA3', 'BEEF3',
       'CMIG4', 'COGN3', 'CPFE3', 'CPLE3', 'CSAN3', 'CSMG3', 'CSNA3', 'CYRE3',
       'DIRR3', 'ECOR3', 'EGIE3', 'EMBJ3', 'ENEV3', 'EQTL3', 'EZTC3', 'FLRY3',
       'GGBR4', 'GOAU4', 'HYPE3', 'ISAE4', 'ITSA4', 'JHSF3', 'KLBN11', 'LREN3',
       'MBRF3', 'MGLU3', 'MOTV3', 'MRVE3', 'MULT3', 'NATU3', 'PETR3', 'PETR4',
       'POMO4', 'PRIO3', 'RADL3', 'RENT3', 'SBSP3', 'SLCE3', 'SMTO3', 'TAEE11',
       'TOTS3', 'UGPA3', 'USIM5', 'VALE3', 'VIVT3', 'WEGE3', 'YDUQ3'],
      dtype='str')
Score Atualizado:  2015
-----
2015 TAMANHOS: Retorno 55, Score PIO 55, Score MF 55
# ------ CRIAÇÃO DO MODELO
## Carteira criada para o ano:  2016
Situação SOLVER:  optimal
2016 -> {'ABEV3': 0.1, 'ALOS3': 0.0, 'ALPA4': 0.0, 'ANIM3': 0.0, 'AXIA3': 0.0, 'AZZA3': 0.0, 'B3SA3': 0.0, 'BEEF3': 0.0, 'CMIG4': 0.0, 'COGN3': 0.1, 'CPFE3': 0.0, 'CPLE3': 0.0, 'CSAN3': 0.0, 'CSMG3': 0.0, 'CSN

In [7]:
carteiras_anuais

{2016: {'pesos': {'ABEV3': 0.1,
   'ALOS3': 0.0,
   'ALPA4': 0.0,
   'ANIM3': 0.0,
   'AXIA3': 0.0,
   'AZZA3': 0.0,
   'B3SA3': 0.0,
   'BEEF3': 0.0,
   'CMIG4': 0.0,
   'COGN3': 0.1,
   'CPFE3': 0.0,
   'CPLE3': 0.0,
   'CSAN3': 0.0,
   'CSMG3': 0.0,
   'CSNA3': 0.0,
   'CYRE3': 0.0,
   'DIRR3': 0.0,
   'ECOR3': 0.0,
   'EGIE3': 0.1,
   'EMBJ3': 0.0,
   'ENEV3': 0.0,
   'EQTL3': 0.0,
   'EZTC3': 0.0,
   'FLRY3': 0.1,
   'GGBR4': 0.0,
   'GOAU4': 0.0,
   'HYPE3': 0.0,
   'ISAE4': 0.1,
   'ITSA4': 0.0,
   'JHSF3': 0.0,
   'KLBN11': 0.0,
   'LREN3': 0.1,
   'MBRF3': 0.0,
   'MGLU3': 0.0,
   'MOTV3': 0.0,
   'MRVE3': 0.1,
   'MULT3': 0.0,
   'NATU3': 0.0,
   'PETR3': 0.0,
   'PETR4': 0.0,
   'POMO4': 0.0,
   'PRIO3': 0.0,
   'RADL3': 0.1,
   'RENT3': 0.0,
   'SBSP3': 0.0,
   'SLCE3': 0.0,
   'SMTO3': 0.0,
   'TAEE11': 0.1,
   'TOTS3': 0.0,
   'UGPA3': 0.1,
   'USIM5': 0.0,
   'VALE3': 0.0,
   'VIVT3': 0.0,
   'WEGE3': 0.0,
   'YDUQ3': 0.0}},
 2017: {'pesos': {'ABEV3': 0.0,
   'ALOS3': 0.

In [8]:
linhas = []
for an in anos:
    an = int(an.split('-')[0])
    print(an)
    for k, v in carteiras_anuais[an]['pesos'].items():
        if round(v, 4)>0.000 and v <= vb_peso_maximo*1.10:
            linhas.append({'ano': an, 'ativo': k, 'peso': v})
df_portfolios = pd.DataFrame(linhas)

# # visões instantâneas:
# df_portfolios[df_portfolios['ano'] == 2016].sort_values('peso', ascending=False)  # uma carteira
# df_portfolios.pivot(index='ativo', columns='ano', values='peso')                  # matriz ativo × ano
# df_portfolios.groupby('ativo')['ano'].count().sort_values(ascending=False)  


2016
2017
2018
2019
2020
2021
2022
2023
2024
2025
2026


In [9]:
df_portfolios[df_portfolios['ano'] == 2026].sort_values('peso', ascending=False)

,ano,ativo,peso
100,2026,ABEV3,0.1
101,2026,ANIM3,0.1
102,2026,CEAB3,0.1
103,2026,CMIN3,0.1
104,2026,CURY3,0.1
105,2026,CVCB3,0.1
106,2026,JHSF3,0.1
107,2026,LREN3,0.1
108,2026,SAUD3,0.1
109,2026,TIMS3,0.1


In [9]:
df_portfolios.to_csv('../carteiras/goal_programming/carteiras_goal_programming.csv')
